In [38]:
import polars as pl
from polars import col


pl.Config.set_tbl_rows(1000)      # Allow rendering up to 1000 rows inline
pl.Config.set_tbl_cols(20)        # Allow rendering up to 20 columns wide
pl.Config.set_fmt_str_lengths(50) # Prevent text strings from getting cropped


polars.config.Config

In [4]:
trade_schema = pl.Struct([
    pl.Field("e", pl.String),
    pl.Field("E", pl.Int64),
    pl.Field("s", pl.String),
    pl.Field("t", pl.Int64),
    pl.Field("p", pl.String), # Must stay String here to handle quotes
    pl.Field("q", pl.String), # Must stay String here to handle quotes
    pl.Field("T", pl.Int64),
    pl.Field("m", pl.Boolean),
    pl.Field("M", pl.Boolean)
])

df_trades = (
    pl.read_csv(
        "/Users/oceansxyz/alphanode-metal/logs/ingest.jsonl",
        has_header=False,
        new_columns=["raw_line"],
        separator="\n"
    )
    .filter(pl.col("raw_line").str.contains('"e":"trade"'))
    .select([
        # 1. Cast your log timestamp to integer right away
        pl.col("raw_line").str.extract(r"^(\d+)").cast(pl.Int64).alias("log_timestamp"),
        
        # 2. Extract and decode the raw JSON object
        pl.col("raw_line").str.extract(r"(\{.*\})$").str.json_decode(dtype=trade_schema).alias("data")
    ])
    .select([
        pl.col("log_timestamp"),
        # 3. Pull fields and cast them to proper types immediately
        pl.col("data").struct.field("e").alias("event_type"),
        pl.col("data").struct.field("E").alias("event_time"),
        pl.col("data").struct.field("s").alias("symbol"),
        pl.col("data").struct.field("t").alias("trade_id"),
        pl.col("data").struct.field("p").cast(pl.Float64).alias("price"),    # Cast early
        pl.col("data").struct.field("q").cast(pl.Float64).alias("quantity"), # Cast early
        pl.col("data").struct.field("T").alias("trade_time"),
        pl.col("data").struct.field("m").alias("is_buyer_maker")
    ])
)

#1780678107499 {"e":"trade","E":1780678107512,"s":"BTCUSDT","t":233870993,"p":"60982.01000000","q":"0.00492000","T":1780678107512,"m":false,"M":true}

In [5]:
df_trades.head()

log_timestamp,event_type,event_time,symbol,trade_id,price,quantity,trade_time,is_buyer_maker
i64,str,i64,str,i64,f64,f64,i64,bool
1780678107499,"""trade""",1780678107512,"""BTCUSDT""",233870993,60982.01,0.00492,1780678107512,false
1780678107549,"""trade""",1780678107563,"""USDCUSDT""",119541896,1.00035,1082.0,1780678107563,true
1780678107600,"""trade""",1780678107614,"""USDCUSDT""",119541897,1.00035,311.0,1780678107613,true
1780678107600,"""trade""",1780678107614,"""USDCUSDT""",119541898,1.00035,61.0,1780678107613,true
1780678107621,"""trade""",1780678107635,"""USDCUSDT""",119541899,1.00035,9.0,1780678107634,true


In [ ]:
df_trades.filter(col("symbol")=="BTCUSDT")\
    .select(col("log_timestamp"), col("event_time"), col("trade_time"), col("symbol"), col("price"), col("quantity"), col("is_buyer_maker"))\
    .with_columns(datetime = pl.from_epoch("event_time", time_unit="ms"))
    

log_timestamp,event_time,trade_time,symbol,price,quantity,is_buyer_maker,datetime
i64,i64,i64,str,f64,f64,bool,datetime[μs]
1780678107499,1780678107512,1780678107512,"""BTCUSDT""",60982.01,0.00492,false,2026-06-05 16:48:27.512
1780678107702,1780678107715,1780678107715,"""BTCUSDT""",60982.01,0.00019,false,2026-06-05 16:48:27.715
1780678107975,1780678107988,1780678107988,"""BTCUSDT""",60982.01,0.00048,false,2026-06-05 16:48:27.988
1780678108122,1780678108136,1780678108135,"""BTCUSDT""",60974.01,0.00032,false,2026-06-05 16:48:28.136
1780678108297,1780678108310,1780678108309,"""BTCUSDT""",60974.01,0.00081,false,2026-06-05 16:48:28.310
…,…,…,…,…,…,…,…
1780803075021,1780803075104,1780803075103,"""BTCUSDT""",61636.08,0.01,false,2026-06-07 03:31:15.104
1780803075139,1780803075227,1780803075226,"""BTCUSDT""",61636.08,0.00759,false,2026-06-07 03:31:15.227
1780803076016,1780803076103,1780803076102,"""BTCUSDT""",61640.0,0.0093,false,2026-06-07 03:31:16.103


In [8]:
df_trades.shape

(2287277, 9)

In [24]:
df_trades.filter(pl.col("event_type")!="trade")

log_timestamp,event_type,event_time,symbol,trade_id,price,quantity,trade_time,is_buyer_maker
i64,str,i64,str,i64,f64,f64,i64,bool


In [17]:
#1780678108431 {"lastUpdateId":1391613837,

# "bids":[["60951.74000000","0.56900000"],["60948.99000000","0.02299000"],["60948.98000000","0.07702000"],
# ["60948.65000000","0.02717000"],["60948.64000000","0.02207000"]],

# "asks":[["60951.75000000","0.25156000"],["60955.52000000","0.00019000"],["60958.27000000","0.06895000"],
# ["60958.68000000","0.00013000"],["60960.35000000","0.16100000"]]}


In [21]:
# 1. Define the specific layout of the orderbook structure
orderbook_schema = pl.Struct([
    pl.Field("lastUpdateId", pl.Int64),
    pl.Field("bids", pl.List(pl.List(pl.String))), 
    pl.Field("asks", pl.List(pl.List(pl.String)))  
])

# 2. Read as a fast flat-text file first to bypass invalid starting characters
df_orderbooks = (
    pl.read_csv(
        "/Users/oceansxyz/alphanode-metal/logs/ingest.jsonl", 
        has_header=False, 
        new_columns=["raw_line"], 
        separator="\n"
    )
    # Using lazy execution after initial load for optimization
    .lazy()
    
    # Isolate only orderbook lines safely using standard string filtering
    .filter(pl.col("raw_line").str.contains('"lastUpdateId"'))
    
    # Split the raw line: Everything before the first '{' is timestamp text, everything else is the JSON payload
    .select([
        pl.col("raw_line").str.splitn("{", 2).struct.field("field_0").str.strip_chars().cast(pl.Int64).alias("timestamp"),
        # Re-attach the splitting curly brace to build a valid JSON string
        (pl.lit("{") + pl.col("raw_line").str.splitn("{", 2).struct.field("field_1")).str.json_decode(dtype=orderbook_schema).alias("data")
    ])
    
    # Clean the types and map columns
    .select([
        pl.from_epoch(pl.col("timestamp"), time_unit="ms").dt.replace_time_zone("UTC").alias("timestamp"),
        pl.col("data").struct.field("lastUpdateId"),
        pl.concat_list([
            pl.col("data").struct.field("bids").list.eval(pl.struct([
                pl.element().list.get(0).cast(pl.Float64).alias("price"),
                pl.element().list.get(1).cast(pl.Float64).alias("quantity"),
                pl.lit("bid").alias("side")
            ])),
            pl.col("data").struct.field("asks").list.eval(pl.struct([
                pl.element().list.get(0).cast(pl.Float64).alias("price"),
                pl.element().list.get(1).cast(pl.Float64).alias("quantity"),
                pl.lit("ask").alias("side")
            ]))
        ]).alias("orders")
    ])
    .explode("orders")
    .unnest("orders")
    .collect()
)


In [22]:
df_orderbooks

timestamp,lastUpdateId,price,quantity,side
"datetime[μs, UTC]",i64,f64,f64,str
2026-06-05 16:48:28.006 UTC,4487691895,60982.0,0.19394,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60981.99,0.00081,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60980.64,0.00009,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60980.42,0.00458,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60980.08,0.00009,"""bid"""
…,…,…,…,…
2026-06-07 03:31:17.350 UTC,190933403,1.00017,1.924513e6,"""ask"""
2026-06-07 03:31:17.350 UTC,190933403,1.00018,2.138907e6,"""ask"""
2026-06-07 03:31:17.350 UTC,190933403,1.00019,1.703468e6,"""ask"""


In [28]:
df_orderbooks.filter(col("price")>50000).head(10)

timestamp,lastUpdateId,price,quantity,side
"datetime[μs, UTC]",i64,f64,f64,str
2026-06-05 16:48:28.006 UTC,4487691895,60982.0,0.19394,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60981.99,0.00081,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60980.64,0.00009,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60980.42,0.00458,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60980.08,0.00009,"""bid"""
2026-06-05 16:48:28.006 UTC,4487691895,60982.01,2.05686,"""ask"""
2026-06-05 16:48:28.006 UTC,4487691895,60982.02,0.00018,"""ask"""
2026-06-05 16:48:28.006 UTC,4487691895,60983.54,0.00084,"""ask"""
2026-06-05 16:48:28.006 UTC,4487691895,60983.74,0.00009,"""ask"""


In [43]:
df_orderbooks.filter(col("price")>50000)\
    .filter(col("side")=="bid")\
    .with_columns(row_id = pl.col("timestamp").rank("dense"))\
        .filter(col("row_id").is_in([1,2,3,4])).head(20)

timestamp,lastUpdateId,price,quantity,side,row_id
"datetime[μs, UTC]",i64,f64,f64,str,u32
2026-06-05 16:48:28.006 UTC,4487691895,60982.0,0.19394,"""bid""",1
2026-06-05 16:48:28.006 UTC,4487691895,60981.99,0.00081,"""bid""",1
2026-06-05 16:48:28.006 UTC,4487691895,60980.64,0.00009,"""bid""",1
2026-06-05 16:48:28.006 UTC,4487691895,60980.42,0.00458,"""bid""",1
2026-06-05 16:48:28.006 UTC,4487691895,60980.08,0.00009,"""bid""",1
2026-06-05 16:48:28.431 UTC,1391613837,60951.74,0.569,"""bid""",2
2026-06-05 16:48:28.431 UTC,1391613837,60948.99,0.02299,"""bid""",2
2026-06-05 16:48:28.431 UTC,1391613837,60948.98,0.07702,"""bid""",2
2026-06-05 16:48:28.431 UTC,1391613837,60948.65,0.02717,"""bid""",2


In [36]:
zst_path ="/Users/oceansxyz/alphanode-metal/logs/ingest/ingest_20260608_03.jsonl.zst"

In [37]:
df_zst = pl.read_ndjson(zst_path)

df_zst.head()

recv_ts_utc_ms,symbol,kind,ts_ms,trade_px,trade_qty
i64,str,str,i64,f64,f64
1780890563237,"""USDCUSDT""","""trade""",1780890563207,1.00022,670.0
1780890563239,"""USDCUSDT""","""trade""",1780890563207,1.00022,1341.0
1780890563613,"""USDCUSDT""","""trade""",1780890563581,1.00022,134.0
1780890563643,"""BTCUSDT""","""trade""",1780890563612,63109.99,0.00569
1780890563844,"""USDCUSDT""","""trade""",1780890563814,1.00022,23.0


In [38]:
df_zst.write_parquet("/Users/oceansxyz/alphanode-metal/logs/ingest/ingest_20260608_08.parquet", compression="zstd")